In [ ]:
from config import config
# Standard library imports
import os
import io
import re
import glob
import json
import math
import time
import shutil
import pickle
import base64
import random
import logging
import warnings
import concurrent.futures
import threading
import queue
import types
from pathlib import Path
from datetime import datetime
from functools import partial
from typing import Dict, Any, List, Set, Tuple, Optional, Union
from dataclasses import dataclass
from collections import Counter, defaultdict
from multiprocessing import Pool, cpu_count

# Third-party imports
# Data processing and analysis
import numpy as np
import pandas as pd
from pandas import DataFrame, Series
from tabulate import tabulate
import seaborn as sns
import matplotlib.pyplot as plt

# Progress tracking
from tqdm import tqdm
from tqdm.auto import tqdm

# HTTP requests
import requests
import backoff

# Natural language processing
import nltk
from nltk.tokenize import sent_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.util import ngrams
nltk.download('punkt')
nltk.download('punkt_tab')

import spacy
from sacrebleu.metrics import CHRF

# Deep learning and model-related
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from transformers import (
    pipeline,
    AutoModel,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    GPT2LMHeadModel,
    GPT2Tokenizer,
    BitsAndBytesConfig,
)
from sentence_transformers import SentenceTransformer, CrossEncoder
import bitsandbytes
import accelerate

# LLMs integration
from openai import OpenAI
from anthropic import Anthropic
from huggingface_hub import login
import replicate
os.environ["REPLICATE_API_TOKEN"] = config['key']['REPLICATE_API_TOKEN']
from google import genai

# Metrics and evaluation
from evaluate import load
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from scipy.spatial.distance import cosine
from scipy.stats import (
    pearsonr,
    spearmanr,
    mannwhitneyu,
    entropy,
    ks_2samp,
    ttest_ind,
    f_oneway,
)
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Custom exceptions and types
class DataProcessingError(Exception):
    """Custom exception for data processing errors"""
    pass

class ModelError(Exception):
    """Custom exception for model-related errors"""
    pass

# Type definitions
ProcessedData = Dict[str, Union[str, float, List[Dict[str, Any]]]]
MetricsResult = Dict[str, Union[float, Dict[str, float]]]

# Environment variables
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('pipeline.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

## Reddit QA Generation

In [ ]:
import openai
import json
from typing import Dict, Any

class QCAMetricAnalyzer:
    def __init__(self):
        """Initialize the QCA metric analyzer with OpenAI client"""
        self.client = OpenAI(api_key=config['key']['OPENAI_API_KEY'])
        # self.model = "gpt-4o-mini"
        self.model = "model"
        
        # Define requirement prompts based on the QCA generation guidelines
        self.requirement_prompts = {
            "fact_based_single_answer": {
                "requirement": "fact_based_single_answer",
                "prompt": """Determine whether the question has **exactly one correct, fact-based answer**.

Question: {question}
Context: {context}
Answer: {answer}

A question meets this requirement if:
- It has one **factual**, **verifiable** answer
- It is **not** subjective, opinion-based, or open-ended
- It does **not** depend on user preferences, multiple viewpoints, or recommendations

Respond in JSON format:
{{"requirement": "fact_based_single_answer", "meetornot": "yes"}} or
{{"requirement": "fact_based_single_answer", "meetornot": "no"}}"""
            },
            
            "no_subjectivity": {
                "requirement": "no_subjectivity", 
                "prompt": """Does this question avoid subjectivity (feelings, recommendations, opinions)?

Question: {question}

A question meets this requirement if it does NOT ask about:
- How users felt
- What users recommended  
- Subjective opinions or preferences
- Vague "why" questions that invite speculation

Respond in JSON format: {{"requirement": "no_subjectivity", "meetornot": "yes/no"}}"""
            },
            
            "no_meta_reddit": {
                "requirement": "no_meta_reddit",
                "prompt": """Determine whether the question avoids referencing the Reddit post or discussion **as a source**.

Question: {question}
Context: {context}
Answer: {answer}

The question and context must **not** ask about or refer to:
- What commenters said, thought, or recommended
- What the thread, subreddit, or post discusses
- Summaries of the discussion or opinions in the post
- Any phrasing like “according to the thread/discussion/subreddit/post”

In short, the QA pair should stand alone without relying on Reddit as a quoted source of authority or opinion.

Respond in JSON format:
{{"requirement": "no_meta_reddit", "meetornot": "yes"}} or
{{"requirement": "no_meta_reddit", "meetornot": "no"}}"""
            },

            
            "context_location_time": {
                "requirement": "context_location_time", 
                "prompt": """Determine whether the **combined** question and context provide all of the following elements:
- A specific **date**, **month/year**, or **timeframe**
- A **county** name
- A **state** name

Question: {question}
Context: {context}

If all three elements are present across the question and context together, respond with:
{{"requirement": "context_location_time", "meetornot": "yes"}}
Otherwise, respond with:
{{"requirement": "context_location_time", "meetornot": "no"}}
"""
            },
            
            
            "no_answer_leakage": {
                "requirement": "no_answer_leakage",
                "prompt": """Determine whether the context **avoids directly revealing the answer**.

Question: {question}
Context: {context}
Answer: {answer}

The context meets this requirement if it does **not** explicitly state the answer
 — in other words, the answer cannot be directly extracted or copied from the context without additional reasoning.

Respond in JSON format:
{{"requirement": "no_answer_leakage", "meetornot": "yes"}} or
{{"requirement": "no_answer_leakage", "meetornot": "no"}}"""
            },
            
            
            "no_meta_content": {
                "requirement": "no_meta_content",
                "prompt": """Does this QA pair avoid meta-Reddit content?

Question: {question}
Context: {context} 
Answer: {answer}

The QA pair meets this requirement if it does NOT reference:
- Karma, upvotes, downvotes
- Moderators or moderation
- Usernames or Reddit users
- Reddit-specific terminology

Respond in JSON format: {{"requirement": "no_meta_content", "meetornot": "yes/no"}}"""
            },
            
            "privacy_safety": {
                "requirement": "privacy_safety",
                "prompt": """Does this QA pair respect privacy and safety guidelines?

Question: {question}
Context: {context}
Answer: {answer}

The QA pair meets this requirement if it does NOT contain:
- Personal phone numbers
- Home addresses
- Other personally identifying information
- Doxxing information

Respond in JSON format: {{"requirement": "privacy_safety", "meetornot": "yes/no"}}"""
            },
            "local_grounding": {
                "requirement": "local_grounding",
                "prompt": """Does this QA pair is about local information of {county}, {state}?
Question: {question}
Context: {context}
Answer: {answer}
The QA pair meets this requirement if it:
- References local events, activities, places, or people, etc.
- Uses specific local information/knowledge relevant to {county}, {state}.

Respond in JSON format: {{"requirement": "local_grounding", "meetornot": "yes/no"}}"""

            }
        }
    
    def _query_gpt(self, prompt: str) -> Dict[str, Any]:
        """Query GPT-4o-mini and parse JSON response"""
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a quality checker for QA pairs. Always respond in valid JSON format."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
                max_tokens=100
            )
            
            response_text = response.choices[0].message.content.strip()
            
            # Parse JSON response
            try:
                result = json.loads(response_text)
                return result
            except json.JSONDecodeError:
                # Fallback: extract JSON if wrapped in markdown or other text
                import re
                json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
                if json_match:
                    result = json.loads(json_match.group())
                    return result
                else:
                    print(f"Failed to parse JSON response: {response_text}")
                    return {"requirement": "unknown", "meetornot": "no"}
                    
        except Exception as e:
            print(f"Error querying GPT: {str(e)}")
            return {"requirement": "unknown", "meetornot": "no"}
    
    def analyze_qa_pair(self, question: str, context: str, answer: str, county: str, state: str) -> Dict[str, str]:
        """
        Analyze a QA pair against all requirements
        
        Args:
            question (str): The question text
            context (str): The context text  
            answer (str): The answer text
            
        Returns:
            Dict[str, str]: Dictionary mapping requirement names to yes/no
        """
        results = {}
        
        for req_name, req_config in self.requirement_prompts.items():
            # Format the prompt with the QA pair data
            formatted_prompt = req_config["prompt"].format(
                question=question,
                context=context, 
                answer=answer,
                county=county,
                state=state
            )
            
            # Query GPT for this requirement
            gpt_response = self._query_gpt(formatted_prompt)
            
            # Extract the yes/no result
            meetornot = gpt_response.get("meetornot", "no").lower()
            results[req_name] = "yes" if meetornot == "yes" else "no"
        
        return results
    
    def get_requirement_descriptions(self, county: str, state: str) -> Dict[str, str]:
        """Get human-readable descriptions of each requirement"""
        descriptions = {
            "fact_based_single_answer": "Question MUST have exactly one factual, correct answer that is not subjective, opinion-based, or open-ended. The answer must not depend on personal preferences or multiple viewpoints.",
            "no_subjectivity": "Question MUST NOT ask about user opinions, feelings, preferences, or include vague 'why' questions that invite speculation or interpretation.",
            "no_meta_reddit": "Question MUST NOT ask about what commenters said, what the post/thread/subreddit discusses, or reference Reddit itself (e.g., 'according to the post/thread/comments').",
            "context_location_time": "Context MUST include all four: a specific date/month or timeframe, year, the name of a county, and the name of a state.",
            "no_answer_leakage": "Context MUST NOT explicitly state or allow direct copying of the answer. The answer should require some reasoning beyond repeating the context.",
            "no_meta_content": "QA pair MUST NOT reference Reddit-specific elements such as karma, upvotes, downvotes, moderators, usernames, or Reddit users in general.",
            "privacy_safety": "QA pair MUST NOT contain personal phone numbers, home addresses, identifying information, or anything that could pose a privacy or safety risk.",
            "local_grounding": f"QA pair MUST reflect local grounding by referencing local events, places, people, or other location-specific knowledge relevant to {county}, {state}.",
        }
        return descriptions
        
    def calculate_overall_score(self, results: Dict[str, str]) -> float:
        """Calculate overall compliance score as percentage of requirements met"""
        total_requirements = len(results)
        met_requirements = sum(1 for value in results.values() if value == "yes")
        return met_requirements / total_requirements if total_requirements > 0 else 0.0

In [ ]:
class QCAGenerator:
    def __init__(self, config: Dict[str, Any]):
        """
        Initialize the QCA Generator with configuration
        
        Args:
            config (Dict[str, Any]): Configuration dictionary containing API keys and settings
        """
        self.config = config
        # Remove OpenAI and Replicate clients — only keep Anthropic
        self.anthropic_client = Anthropic(api_key=config['key']['CLAUDE_API_KEY'])
        self.openai_client = OpenAI(api_key=config['key']['OPENAI_API_KEY'])
        
        # Initialize QCA Metric Analyzer
        self.qca_analyzer = QCAMetricAnalyzer()
        
        # Create output directory
        self.output_dir = os.path.join('processed_data', 'qca_output')
        os.makedirs(self.output_dir, exist_ok=True)

        # Setup logging
        self.setup_logging()
        
        # Initialize queues and tracking
        self.result_queue = queue.Queue()
        self.error_queue = queue.Queue()
        self.processed_rows = set()
        
        # Initialize state
        self.last_api_call = 0
        self.api_rate_limit = 1.0  # minimum seconds between API calls

        # Add archive directory for completed checkpoints
        self.archive_dir = os.path.join(self.output_dir, 'archived_checkpoints')
        os.makedirs(self.archive_dir, exist_ok=True)
        
        # Load checkpoint if exists
        self.load_checkpoint()
        
    def setup_logging(self) -> None:
        """Configure logging with file and console handlers"""
        log_format = '%(asctime)s - %(levelname)s - %(message)s'
        logging.basicConfig(
            level=logging.INFO,
            format=log_format,
            handlers=[
                logging.FileHandler(os.path.join(self.output_dir, 'qca_generator.log')),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)

    def load_checkpoint(self) -> None:
        """Load checkpoint of processed rows if exists"""
        checkpoint_path = os.path.join(self.output_dir, 'checkpoint.json')
        if os.path.exists(checkpoint_path):
            try:
                with open(checkpoint_path, 'r') as f:
                    checkpoint_data = json.load(f)
                self.processed_rows = set(checkpoint_data['processed_rows'])
                self.logger.info(f"Loaded checkpoint with {len(self.processed_rows)} processed rows")
            except Exception as e:
                self.logger.error(f"Error loading checkpoint: {str(e)}")

    def save_checkpoint(self) -> None:
        """Save checkpoint of processed rows"""
        checkpoint_path = os.path.join(self.output_dir, 'checkpoint.json')
        try:
            with open(checkpoint_path, 'w') as f:
                json.dump({
                    'processed_rows': list(self.processed_rows),
                    'timestamp': datetime.now().isoformat()
                }, f)
            self.logger.info("Checkpoint saved successfully")
        except Exception as e:
            self.logger.error(f"Error saving checkpoint: {str(e)}")

    def cleanup_checkpoints(self) -> None:
        """
        Archive and clean up checkpoints after category completion
        """
        try:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            checkpoint_path = os.path.join(self.output_dir, 'checkpoint.json')
            
            if os.path.exists(checkpoint_path):
                # Archive current checkpoint
                archive_name = f'checkpoint.json'
                archive_path = os.path.join(self.archive_dir, archive_name)
                
                shutil.copy2(checkpoint_path, archive_path)
                os.remove(checkpoint_path)
                
                self.logger.info(f"Checkpoint archived to: {archive_name}")
                
            # Reset processed rows
            self.processed_rows = set()
            
        except Exception as e:
            self.logger.error(f"Error cleaning up checkpoints: {str(e)}")

    def cleanup_logs(self) -> None:
        """
        Archive logs after category completion
        """
        try:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            log_path = 'qca_generator.log'
            
            if os.path.exists(log_path):
                # Archive current log
                archive_name = f'qca_generator.log'
                archive_path = os.path.join(self.archive_dir, archive_name)
                
                shutil.copy2(log_path, archive_path)
                
                # Clear current log file
                with open(log_path, 'w') as f:
                    f.write(f"Log reset after processing dataset\n")
                
                self.logger.info(f"Logs archived to: {archive_name}")
                
        except Exception as e:
            self.logger.error(f"Error cleaning up logs: {str(e)}")

    def cleanup_progress_files(self) -> None:
        """
        Clean up intermediate progress files
        """
        try:
            progress_files = [f for f in os.listdir(self.output_dir) 
                              if f.startswith(f'qca_progress_')]
            
            for file in progress_files:
                file_path = os.path.join(self.output_dir, file)
                archive_path = os.path.join(self.archive_dir, file)
                
                shutil.move(file_path, archive_path)
                
            self.logger.info(f"Archived {len(progress_files)} progress files")
            
        except Exception as e:
            self.logger.error(f"Error cleaning up progress files: {str(e)}")

    @backoff.on_exception(
        backoff.expo,
        Exception,
        max_tries=3,
        on_backoff=lambda details: logging.warning(f"Retry attempt {details['tries']}")
    )
    def generate_prompt(self, row: pd.Series) -> str:
        """Generate the QAGenPrompt for each row based on refined guidelines."""
        return f"""
    =============================================================================
    LOCAL QA GENERATION PROMPT 
    =============================================================================
    You are given a single Reddit thread (post + comments) from a *local* subreddit of {row['county']}, {row['state']}. 
    From it, you will generate ONE QA pair that is:

        • Clear, locally grounded about {row['county']}, {row['state']}, and answerable with one correct response.
        • Either:
            ◦ Track A — Fact-based Local QA → grounded in verifiable, non-subjective local information
            ◦ Track B — Community Insight QA → reflects clearly shared local community experiences

    Use the structure below. Do NOT fabricate information.
    ─────────────────────────────────────────────────────────────────────────────
    STRICT QA RULES
    ─────────────────────────────────────────────────────────────────────────────
    1. **NO THREAD-REFERENTIAL QUESTIONS.**  
    • The QA pair should stand alone without relying on Reddit as a quoted source of authority or opinion.
    • Forbidden phrases include: “was mentioned as…”, “according to the thread/
        discussion/Reddit post/comment/commenter”, “what did commenters think”, etc.  
    • Litmus test: *Would the question still make perfect sense to someone who
        never sees the thread?* If not, reject it.

    2. FACT-BASED ONLY: Question MUST has **exactly one correct, fact-based answer.
    • Prefer using the post's original question if applicable.  
    • Include time/place qualifiers for precision.
    • It does **not** depend on user preferences, multiple viewpoints, or recommendations

    3. AVOID all of these:
    • Questions asking how people felt, or vague "why" questions.
    • Any framing that implies subjectivity, speculation, or unconfirmed information.

    3. CONTEXT must:
    • Be 2–4 neutral sentences summarizing thread purpose.
    • Mention exact date (or month), year, county, and state.
    • Attribute information sources.
    • Do **not** leak answer.

    4. ANSWER must:
    • Track A: one exact, verifiable fact (from post or multiple comments).
    • Track B: a consensus summary if multiple users echoed same view (no "commenters said" or "commenters recommand", etc.)).

    5. Prefer Track A. Use Track B only if consensus is unambiguous.
    
    6. No meta-Reddit content (karma, mods, usernames).  
    
    7. Privacy & Safety: no doxxing, no private phone numbers.

    ─────────────────────────────────────────────────────────────────────────────
    THREAD DETAILS
    ─────────────────────────────────────────────────────────────────────────────
    Post Title: {row['title']}

    Post Information:
    - Date: {row['created_time']}
    - County: {row['county']}
    - State: {row['state']}

    Post Content:
    {row['selftext']}

    Comments:
    {row['comments']}

    ─────────────────────────────────────────────────────────────────────────────
    OUTPUT FORMAT
    ─────────────────────────────────────────────────────────────────────────────
    [PAIR1]
    Question: <one clearly answerable question>
    Context: <neutral summary with time and location>
    Answer: <fact or consensus summary>
    Selected Comments: <e.g., 3, 7, 12>
    Pair_type: <fact|insight>

    ─────────────────────────────────────────────────────────────────────────────
    EXAMPLES OF ACCEPTABLE QUESTIONS
    ─────────────────────────────────────────────────────────────────────────────
    • What city program was in contract phase in March 2024 in Broomfield, Colorado?
    • Which consulting firm was hired in June 2025 to evaluate Athens' zoning?
    • What fungus linked to bird feces raised public health concern in Honolulu?
    ✘ Bad questions that are not acceptable: What do residents want? What do recent discussions say? Which business do people like? What do some suggest?

    ─────────────────────────────────────────────────────────────────────────────
    HARD STOPS
    ─────────────────────────────────────────────────────────────────────────────
    ✘ Do NOT generate a QA if:
        - The answer is not clearly stated or verifiable
        - No single correct answer can be derived
        - Content is entirely opinion-based or speculative

    =============================================================================
        """

    @backoff.on_exception(backoff.expo, Exception, max_tries=3)
    def get_llm_response(self, messages: List[Dict[str, str]], use_model, use_claude: bool = True) -> str:
        """
        Get response from LLM API with rate limiting
        
        Args:
            messages (List[Dict[str, str]]): Conversation messages
            use_claude (bool): Whether to use Claude (True) or OpenAI (False)
            
        Returns:
            str: LLM response
        """
        # Simple rate limiting
        elapsed = time.time() - self.last_api_call
        if elapsed < self.api_rate_limit:
            time.sleep(self.api_rate_limit - elapsed)
            
        try:
            if use_claude:
                # Convert messages format for Claude (remove system messages)
                user_content = ""
                for msg in messages:
                    if msg["role"] == "user":
                        user_content += f"{msg['content']}\n\n"
                    elif msg["role"] == "assistant":
                        user_content += f"Previous response:\n{msg['content']}\n\n"
                
                response = self.anthropic_client.messages.create(
                    model=use_model,
                    temperature=0.0,
                    max_tokens=256,
                    messages=[{"role": "user", "content": user_content.strip()}]
                )
                self.last_api_call = time.time()
                return response.content[0].text
            else:
                response = self.openai_client.chat.completions.create(
                    # model="ft:gpt-4o-2024-08-06:personal:qageneration:BlKBl3ld",
                    # model="o3",
                    model=use_model,
                    messages=messages,
                    temperature=0.7,
                    max_tokens=200,
                )
                self.last_api_call = time.time()
                return response.choices[0].message.content.strip()
            
        except Exception as e:
            self.logger.error(f"Error getting LLM response: {str(e)}")
            raise

    def parse_llm_response(self, response: str) -> Optional[Dict[str, str]]:
        """
        Parse LLM response and return only the first QCA pair.
        
        Args:
            response (str): Raw response from LLM
            
        Returns:
            Optional[Dict[str, str]]: The first parsed QCA pair or None if parsing fails
        """
        try:
            current_pair = {}
            
            # Basic splitting logic
            for line in response.split('\n'):
                line = line.strip()
                if not line:
                    continue
                    
                if '[PAIR' in line and current_pair:
                    # Return the first found pair immediately
                    return current_pair
                    
                elif line.startswith('Question:'):
                    current_pair['question'] = line[len('Question:'):].strip()
                elif line.startswith('Context:'):
                    current_pair['context'] = line[len('Context:'):].strip()
                elif line.startswith('Answer:'):
                    current_pair['answer'] = line[len('Answer:'):].strip()
                elif line.startswith('Selected Comments:'):
                    current_pair['selected_comments'] = line[len('Selected Comments:'):].strip()
                elif line.startswith('Pair_type:'):
                    pair_type = line[len('pair_type:'):].strip()
                    current_pair['pair_type'] = pair_type
            
            # Return the first valid pair if one exists
            if current_pair:
                return current_pair

            self.logger.warning("No valid QCA pair found in LLM response")
            return None

        except Exception as e:
            self.logger.error(f"Error parsing LLM response: {str(e)}")
            return None

    def create_refinement_message(self, failed_requirements: List[str], state, county) -> str:
        """
        Create a user message explaining which requirements weren't met
        
        Args:
            failed_requirements (List[str]): List of requirement names that failed
            
        Returns:
            str: Refinement message for the LLM
        """

        requirement_explanations = self.qca_analyzer.get_requirement_descriptions(county, state)
        
        message = "The QA pair you provided does not meet the following requirements:\n\n"
        
        for req in failed_requirements:
            if req in requirement_explanations:
                message += f"• {req.replace('_', ' ').title()}: {requirement_explanations[req]}\n"
        
        message += "\nPlease revise the QA pair to meet all requirements. Keep your output in the same format and only have these content:\n\n"
        message += "[PAIR1]\nQuestion: <revised question>\nContext: <revised context>\nAnswer: <revised answer>\nSelected Comments: <comment numbers>\nPair_type: <fact|insight>"
        
        return message
        
    def process_single_row(self, row_data: Tuple[int, pd.Series]) -> Optional[Dict[str, Any]]:
        """
        Process a single row of data: generate prompt, call LLM, parse response,
        and iteratively refine using QCAMetricAnalyzer if requirements are not met.
        
        Args:
            row_data (Tuple[int, pd.Series]): (index, row)
            
        Returns:
            Optional[Dict[str, Any]]: Result dictionary with QCA pairs, or None on error.
        """
        idx, row = row_data
        try:
            # Skip if already processed
            if idx in self.processed_rows:
                self.logger.info(f"Skipping already processed row {idx}")
                return None

            base_prompt = self.generate_prompt(row)
            result = {'idx': idx}
            
            # Initialize conversation
            messages = [
                {"role": "system", "content": "You are a helpful assistant that generates high-quality question-context-answer pairs from local Reddit threads."},
                {"role": "user", "content": base_prompt}
            ]
            
            attempt = 0
            max_attempts = 3
            
            while attempt < max_attempts:
                try:
                    use_model = "o3"
                    use_claude = False
                    print(f"Using model: {use_model}, Claude: {use_claude}")
                    llm_raw_response = self.get_llm_response(messages, use_model, use_claude)
                    result[f'llm_raw_response_attempt_{attempt + 1}'] = llm_raw_response
                    # print(f"LLM raw response for row {idx}, attempt {attempt + 1}:\n{llm_raw_response}\n")
                    
                    # Parse response
                    pair = self.parse_llm_response(llm_raw_response)

                    if not pair or not isinstance(pair, dict):
                        self.logger.warning(f"No valid QCA pair found for row {idx}, attempt {attempt + 1}. Retrying...")
                        attempt += 1
                        continue
                    
                    # Extract fields safely
                    question = pair.get('question', '')
                    context = pair.get('context', '')
                    answer = pair.get('answer', '')
                    selected_comments = pair.get('selected_comments', '')
                    pair_type = pair.get('pair_type', 'fact')
                    
                    # Check requirements using QCAMetricAnalyzer
                    try:
                        requirements_results = self.qca_analyzer.analyze_qa_pair(
                            question=question,
                            context=context,
                            answer=answer,
                            state=row['state'],
                            county=row['county'],
                            # selected_comments=selected_comments
                        )

                        # print(f"Row {idx} - Requirements results: {requirements_results}")
                        
                        # Check if any requirement failed
                        failed_requirements = [req for req, result_val in requirements_results.items() 
                                             if result_val.lower() == "no"]
                        
                        if len(failed_requirements) < 1:
                            result['QAgenprompt'] = base_prompt
                            result['claude_Question_1'] = question
                            result['claude_QuestionContext_1'] = context
                            result['claude_Answer_1'] = answer
                            result['claude_SelectedComments_1'] = selected_comments
                            result['claude_PairType_1'] = pair_type
                            result['requirements_results'] = str(requirements_results)
                            result['failed_requirements_count'] = len(failed_requirements)
                            
                            self.processed_rows.add(idx)
                            tolerance_msg = "with perfect requirements" if len(failed_requirements) == 0 else f"with {len(failed_requirements)} tolerated failed requirement: {failed_requirements}"
                            self.logger.info(f"Row {idx} completed successfully after {attempt + 1} attempt(s) {tolerance_msg}")
                            return result
                        else:
                            # Too many requirements failed - create refinement message
                            self.logger.info(f"Row {idx}, attempt {attempt + 1}: {len(failed_requirements)} failed requirements (tolerance threshold: 1): {failed_requirements}")
                            
                            # Add the assistant's response to conversation
                            messages.append({"role": "assistant", "content": llm_raw_response})
                            
                            # Add refinement request
                            refinement_message = self.create_refinement_message(failed_requirements, row['state'], row['county'])
                            messages.append({"role": "user", "content": refinement_message})
                            
                            # Store failed requirements for debugging
                            result[f'failed_requirements_attempt_{attempt + 1}'] = str(failed_requirements)
                            result[f'requirements_results_attempt_{attempt + 1}'] = str(requirements_results)
                            
                    except Exception as e:
                        self.logger.error(f"Error in QCA analysis for row {idx}, attempt {attempt + 1}: {str(e)}")
                        # Continue to next attempt on analysis error
                        
                except Exception as e:
                    self.logger.error(f"Error with LLM model for row {idx}, attempt {attempt + 1}: {str(e)}")

                attempt += 1
            
            self.logger.warning(f"Skipping row {idx} after {max_attempts} attempts - more than 1 requirement failed (tolerance threshold exceeded)")
            return None
        
        except Exception as e:
            self.logger.error(f"Error processing row {idx}: {str(e)}")
            self.error_queue.put((idx, str(e)))
        return None

    def process_batch(self, batch_data: List[Tuple[int, pd.Series]]) -> None:
        """
        Process a batch of rows in parallel and put results in queue
        
        Args:
            batch_data (List[Tuple[int, pd.Series]]): List of (index, row) tuples to process
        """
        # Parallel execution with ThreadPoolExecutor
        with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
            futures = [
                executor.submit(self.process_single_row, row_data) 
                for row_data in batch_data
            ]
            
            for future in concurrent.futures.as_completed(futures):
                try:
                    result = future.result()
                    if result:
                        self.result_queue.put(result)
                except Exception as e:
                    self.logger.error(f"Error in batch processing: {str(e)}")

    def save_progress(self, df: pd.DataFrame, category: str, timestamp: str) -> None:
        """
        Save intermediate progress with error handling
        
        Args:
            df (pd.DataFrame): DataFrame to save
            category (str): Category name (rural/urban)
            timestamp (str): Timestamp for the file name
        """
        try:
            output_path = os.path.join(
                self.output_dir, 
                f'{category}_qca_progress_{timestamp}.parquet'
            )
            df.to_parquet(output_path)
            self.logger.info(f"Progress saved to: {output_path}")
            
            # Save checkpoint
            self.save_checkpoint()
            
        except Exception as e:
            self.logger.error(f"Error saving progress: {str(e)}")

    def process_dataset(self, df: pd.DataFrame, category: str) -> pd.DataFrame:
        """
        Process dataset with batch processing and parallel execution
        
        Args:
            df (pd.DataFrame): Input DataFrame to process
            category (str): Category name (rural/urban)
            
        Returns:
            pd.DataFrame: Processed DataFrame with QCA pairs
        """
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        processed_df = df.copy()
        
        # Initialize columns for QCA pairs and analysis results
        base_columns = ['Question', 'QuestionContext', 'Answer', 'SelectedComments', 'PairType', 'QAgenprompt']
        for i in range(1, 3):
            for col in base_columns:
                processed_df[f'claude_{col}_{i}'] = ''
        
        # Add columns for requirements analysis and tolerance tracking
        processed_df['requirements_results'] = ''
        processed_df['failed_requirements_count'] = 0
        
        # Initialize columns for attempt tracking (up to max_attempts)
        max_attempts = 3
        for attempt in range(1, max_attempts + 1):
            processed_df[f'llm_raw_response_attempt_{attempt}'] = ''
            processed_df[f'failed_requirements_attempt_{attempt}'] = ''
            processed_df[f'requirements_results_attempt_{attempt}'] = ''
        
        # Process in batches
        batch_size = 32
        for start_idx in tqdm(range(0, len(processed_df), batch_size)):
            end_idx = min(start_idx + batch_size, len(processed_df))
            batch_data = list(processed_df.iloc[start_idx:end_idx].iterrows())
            
            # Process batch in parallel
            self.process_batch(batch_data)
            
            # Collect results
            while not self.result_queue.empty():
                result = self.result_queue.get()
                idx = result.pop('idx')
                for col, value in result.items():
                    # Handle different value types safely
                    if isinstance(value, (list, dict)):
                        # Convert complex data structures to string for storage
                        processed_df.at[idx, col] = str(value)
                    else:
                        processed_df.at[idx, col] = value
            
            # Handle any errors
            while not self.error_queue.empty():
                idx, error = self.error_queue.get()
                self.logger.warning(f"Error in row {idx}: {error}")
            
            # Save progress periodically
            if (start_idx + batch_size) % 50 == 0:
                self.save_progress(processed_df, category, timestamp)
        
        # Save final results
        final_path = os.path.join(
            self.output_dir, 
            f'{category}_qca_final_{timestamp}.parquet'
        )
        processed_df.to_parquet(final_path)
        self.logger.info(f"Final results saved to: {final_path}")
        
        # Print statistics
        self.print_statistics(processed_df, category)
        
        return processed_df

    def print_statistics(self, df: pd.DataFrame, category: str) -> None:
        """
        Print basic statistics for the QCA generation
        """
        try:
            print(f"\nProcessing Statistics for {category}:")
            print("-" * 50)
            print(f"Total rows processed: {len(df)}")
            
            # Count how many rows have a non-empty question in at least the first pair
            completed_rows = sum(df['claude_Question_1'].astype(bool))
            success_rate = (completed_rows / len(df)) * 100
            
            print(f"\nLLM Model Statistics:")
            print(f"Successfully completed: {completed_rows}")
            print(f"Success rate: {success_rate:.2f}%")
                
            # Average lengths
            avg_question = df['claude_Question_1'].str.len().mean()
            avg_context = df['claude_QuestionContext_1'].str.len().mean()
            avg_answer = df['claude_Answer_1'].str.len().mean()
                
            print("\nAverage lengths:")
            print(f"Questions: {avg_question:.1f} characters")
            print(f"Context: {avg_context:.1f} characters")
            print(f"Answers: {avg_answer:.1f} characters")
            
            # Requirements analysis statistics
            if 'requirements_results' in df.columns:
                requirements_with_results = df[df['requirements_results'] != ''].copy()
                if len(requirements_with_results) > 0:
                    print(f"\nRequirements Analysis:")
                    print(f"Rows with requirements analysis: {len(requirements_with_results)}")
                
            self.logger.info(f"Processing completed for {category}")
            
        except Exception as e:
            self.logger.error(f"Error calculating statistics: {str(e)}")

def main():
    """Main execution function"""
    # Initialize generator
    generator = QCAGenerator(config)
    generator.cleanup_checkpoints()
    generator.cleanup_logs()
    generator.cleanup_progress_files()
    
    try:
        parquet_input_path = 'parquet_input_path'
        df = pd.read_parquet(parquet_input_path)
        df = df[df['comments'].str.len() > 0].reset_index(drop=True)
        df = df.drop_duplicates(subset=['title', 'selftext', 'comments']).reset_index(drop=True)
        print(f"Total rows in input data: {len(df)}")

        # Load and process datasets
        results = {}

        results['subreddits'] = generator.process_dataset(df, 'subreddits')
        
        logging.info(f"Completed processing and cleanup for {'subreddits'} dataset")
        
        return results['subreddits']
        
    except Exception as e:
        logging.error(f"Error in main execution: {str(e)}")
        raise

def run_qca_generation():
    """Entry point function with error handling"""
    try:
        logging.info("Starting QCA generation process...")
        processed_subreddits = main()
        logging.info("QCA generation completed successfully")
        return processed_subreddits
        
    except Exception as e:
        logging.error(f"Fatal error in QCA generation: {str(e)}")
        raise

processed_subreddits = run_qca_generation()

## News QA Generation

In [ ]:
import openai
import json
from typing import Dict, Any

class QCAMetricAnalyzer:
    def __init__(self, openai_api_key: str):
        """Initialize the QCA metric analyzer with OpenAI client"""
        self.client = openai.OpenAI(api_key=openai_api_key)
        # self.model = "gpt-4o-mini"
        self.model = "model"
        
        self.requirement_prompts = {
            "fact_based_single_answer": {
                "requirement": "fact_based_single_answer",
                "prompt": """Determine whether the question has **exactly one correct, fact-based answer**.

Question: {question}
Context: {context}
Answer: {answer}

A question meets this requirement if:
- It has one **factual**, **verifiable** answer based on the news article
- It is **not** subjective, opinion-based, or open-ended
- It does **not** depend on personal preferences, multiple viewpoints, or recommendations
- The answer is directly supported by factual information from the local news article

Respond in JSON format:
{{"requirement": "fact_based_single_answer", "meetornot": "yes"}} or
{{"requirement": "fact_based_single_answer", "meetornot": "no"}}"""
            },
            
            "no_subjectivity": {
                "requirement": "no_subjectivity", 
                "prompt": """Does this question avoid subjectivity (feelings, recommendations, opinions)?

Question: {question}

A question meets this requirement if it does NOT ask about:
- How people felt about local events
- What residents recommended or suggested
- Subjective opinions or preferences about local issues
- Vague "why" questions that invite speculation about community sentiments

Respond in JSON format: {{"requirement": "no_subjectivity", "meetornot": "yes/no"}}"""
            },
            
            "no_meta_news": {
                "requirement": "no_meta_news",
                "prompt": """Determine whether the question avoids referencing the news article **as a source**.

Question: {question}
Context: {context}
Answer: {answer}

The question and context must **not** ask about or refer to:
- What the article reports, says, or discusses
- What journalists wrote or investigated
- Summaries of the article's content or structure
- Any phrasing like "according to the article/report/news story"
- The article itself as the subject of inquiry

The QA pair should focus on the LOCAL EVENTS/FACTS described in the article, not the article as a document.

Respond in JSON format:
{{"requirement": "no_meta_news", "meetornot": "yes"}} or
{{"requirement": "no_meta_news", "meetornot": "no"}}"""
            },
            
            "context_location_time": {
                "requirement": "context_location_time", 
                "prompt": """Determine whether the **combined** question and context provide all of the following elements:
- A specific **date**, **month/year**, or **timeframe** when the local event occurred
- A **county** name where the event took place
- A **state** name where the event took place

Question: {question}
Context: {context}

If all three elements are present across the question and context together, respond with:
{{"requirement": "context_location_time", "meetornot": "yes"}}
Otherwise, respond with:
{{"requirement": "context_location_time", "meetornot": "no"}}
"""
            },
                
            "no_answer_leakage": {
                "requirement": "no_answer_leakage",
                "prompt": """Determine whether the context **avoids directly revealing the answer**.

Question: {question}
Context: {context}
Answer: {answer}

The context meets this requirement if it does **not** explicitly state the answer
 — in other words, the answer cannot be directly extracted or copied from the context without additional reasoning about the local situation.

Respond in JSON format:
{{"requirement": "no_answer_leakage", "meetornot": "yes"}} or
{{"requirement": "no_answer_leakage", "meetornot": "no"}}"""
            },
            
            "verifiable_answer": {
                "requirement": "verifiable_answer",
                "prompt": """Is this answer one exact, verifiable fact from the local news article?

Answer: {answer}
Context: {context}

An answer meets this requirement if it is:
- A specific, factual statement about local events, people, places, or decisions
- Verifiable from the news article content
- Not vague, ambiguous, or speculative
- Contains concrete details (names, dates, locations, specific actions)

Respond in JSON format: {{"requirement": "verifiable_answer", "meetornot": "yes/no"}}"""
            },
            
            "no_meta_content": {
                "requirement": "no_meta_content",
                "prompt": """Does this QA pair avoid meta-journalism content?

Question: {question}
Context: {context} 
Answer: {answer}

The QA pair meets this requirement if it does NOT reference:
- Journalists, reporters, or news staff by name
- Publication details, newspaper names, or media outlets
- Article writing, reporting process, or journalism methodology
- News-specific terminology about coverage or reporting

The focus should be on LOCAL EVENTS and FACTS, not the news reporting process.

Respond in JSON format: {{"requirement": "no_meta_content", "meetornot": "yes/no"}}"""
            },
            
            "privacy_safety": {
                "requirement": "privacy_safety",
                "prompt": """Does this QA pair respect privacy and safety guidelines?

Question: {question}
Context: {context}
Answer: {answer}

The QA pair meets this requirement if it does NOT contain:
- Personal phone numbers or email addresses
- Specific home addresses (street addresses)
- Social security numbers or other identifying numbers
- Information that could enable stalking or harassment
- Private personal details about individuals

General location information (city, county, state) and public official information is acceptable.

Respond in JSON format: {{"requirement": "privacy_safety", "meetornot": "yes/no"}}"""
            },
            
            "local_grounding": {
                "requirement": "local_grounding",
                "prompt": """Does this QA pair demonstrate local grounding for {county}, {state}?

Question: {question}
Context: {context}
Answer: {answer}

The QA pair meets this requirement if it:
- References specific local events, activities, places, or people in {county}, {state}
- Uses local knowledge relevant to the community (local businesses, landmarks, officials)
- Demonstrates understanding of local context and issues
- Contains information that would be particularly meaningful to {county}, {state} residents

Respond in JSON format: {{"requirement": "local_grounding", "meetornot": "yes/no"}}"""
            }
        }
    
    def _query_gpt(self, prompt: str) -> Dict[str, Any]:
        """Query GPT-4o-mini and parse JSON response"""
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a quality checker for QA pairs generated from local news articles. Always respond in valid JSON format."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
                max_tokens=100
            )
            
            response_text = response.choices[0].message.content.strip()
            
            # Parse JSON response
            try:
                result = json.loads(response_text)
                return result
            except json.JSONDecodeError:
                # Fallback: extract JSON if wrapped in markdown or other text
                import re
                json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
                if json_match:
                    result = json.loads(json_match.group())
                    return result
                else:
                    print(f"Failed to parse JSON response: {response_text}")
                    return {"requirement": "unknown", "meetornot": "no"}
                    
        except Exception as e:
            print(f"Error querying GPT: {str(e)}")
            return {"requirement": "unknown", "meetornot": "no"}
    
    def analyze_qa_pair(self, question: str, context: str, answer: str, county: str = "", state: str = "") -> Dict[str, str]:
        """
        Analyze a QA pair against all requirements
        
        Args:
            question (str): The question text
            context (str): The context text  
            answer (str): The answer text
            county (str): County name for local grounding check
            state (str): State name for local grounding check
            
        Returns:
            Dict[str, str]: Dictionary mapping requirement names to yes/no
        """
        results = {}
        
        for req_name, req_config in self.requirement_prompts.items():
            # Format the prompt with the QA pair data
            formatted_prompt = req_config["prompt"].format(
                question=question,
                context=context, 
                answer=answer,
                county=county,
                state=state
            )
            
            # Query GPT for this requirement
            gpt_response = self._query_gpt(formatted_prompt)
            
            # Extract the yes/no result
            meetornot = gpt_response.get("meetornot", "no").lower()
            results[req_name] = "yes" if meetornot == "yes" else "no"
        
        return results
    
    def get_requirement_descriptions(self, county: str = "", state: str = "") -> Dict[str, str]:
        """Get human-readable descriptions of each requirement"""
        location_desc = f" for {county}, {state}" if county and state else ""
        
        descriptions = {
            "fact_based_single_answer": "Question MUST have exactly one factual, verifiable answer based on the news article that is not subjective, opinion-based, or open-ended",
            "no_subjectivity": "Question MUST NOT ask about residents' feelings, recommendations, opinions, or include vague 'why' questions that invite speculation about community sentiments",
            "no_meta_news": "Question MUST NOT ask about what the article reports/says or reference the news article itself as a source (e.g., 'according to the article')",
            "context_location_time": "Context MUST include specific date/month/timeframe, county name, and state name where the local event occurred",
            "no_answer_leakage": "Context MUST NOT explicitly state the answer - the answer should require reasoning about the local situation beyond copying from context",
            "verifiable_answer": "Answer MUST be a specific, factual statement about local events/people/places that is verifiable from the news article and contains concrete details",
            "no_meta_content": "QA pair MUST NOT reference journalists, publication details, reporting process, or journalism methodology - focus on LOCAL EVENTS and FACTS only",
            "privacy_safety": "QA pair MUST NOT contain personal phone numbers, specific home addresses, or information that could enable stalking or harassment",
            "local_grounding": f"QA pair MUST demonstrate local grounding by referencing specific local events, places, people, or community knowledge relevant to the area{location_desc}",
        }
        return descriptions
        
    def calculate_overall_score(self, results: Dict[str, str]) -> float:
        """Calculate overall compliance score as percentage of requirements met"""
        total_requirements = len(results)
        met_requirements = sum(1 for value in results.values() if value == "yes")
        return met_requirements / total_requirements if total_requirements > 0 else 0.0

In [ ]:
class QCAGenerator:
    def __init__(self, config: Dict[str, Any]):
        """
        Initialize the QCA Generator with configuration
        
        Args:
            config (Dict[str, Any]): Configuration dictionary containing API keys and settings
        """
        self.config = config
        # Initialize both clients
        self.anthropic_client = Anthropic(api_key=config['key']['CLAUDE_API_KEY'])
        self.openai_client = OpenAI(api_key=config['key']['OPENAI_API_KEY'])
        
        # Initialize QCA Metric Analyzer with the new implementation
        self.qca_analyzer = QCAMetricAnalyzer(openai_api_key=config['key']['OPENAI_API_KEY'])
        
        # Create output directory
        self.output_dir = os.path.join('processed_data', 'qca_output')
        os.makedirs(self.output_dir, exist_ok=True)

        # Setup logging
        self.setup_logging()
        
        # Initialize queues and tracking
        self.result_queue = queue.Queue()
        self.error_queue = queue.Queue()
        self.processed_rows = set()
        
        # Initialize state
        self.last_api_call = 0
        self.api_rate_limit = 1.0  # minimum seconds between API calls

        # Add archive directory for completed checkpoints
        self.archive_dir = os.path.join(self.output_dir, 'archived_checkpoints')
        os.makedirs(self.archive_dir, exist_ok=True)
        
        # Load checkpoint if exists
        self.load_checkpoint()
        
    def setup_logging(self) -> None:
        """Configure logging with file and console handlers"""
        log_format = '%(asctime)s - %(levelname)s - %(message)s'
        logging.basicConfig(
            level=logging.INFO,
            format=log_format,
            handlers=[
                logging.FileHandler(os.path.join(self.output_dir, 'qca_generator.log')),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)

    def load_checkpoint(self) -> None:
        """Load checkpoint of processed rows if exists"""
        checkpoint_path = os.path.join(self.output_dir, 'checkpoint.json')
        if os.path.exists(checkpoint_path):
            try:
                with open(checkpoint_path, 'r') as f:
                    checkpoint_data = json.load(f)
                self.processed_rows = set(checkpoint_data['processed_rows'])
                self.logger.info(f"Loaded checkpoint with {len(self.processed_rows)} processed rows")
            except Exception as e:
                self.logger.error(f"Error loading checkpoint: {str(e)}")

    def save_checkpoint(self) -> None:
        """Save checkpoint of processed rows"""
        checkpoint_path = os.path.join(self.output_dir, 'checkpoint.json')
        try:
            with open(checkpoint_path, 'w') as f:
                json.dump({
                    'processed_rows': list(self.processed_rows),
                    'timestamp': datetime.now().isoformat()
                }, f)
            self.logger.info("Checkpoint saved successfully")
        except Exception as e:
            self.logger.error(f"Error saving checkpoint: {str(e)}")

    def cleanup_checkpoints(self, category: str) -> None:
        """
        Archive and clean up checkpoints after category completion
        
        Args:
            category (str): Category name being completed
        """
        try:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            checkpoint_path = os.path.join(self.output_dir, 'checkpoint.json')
            
            if os.path.exists(checkpoint_path):
                # Archive current checkpoint
                archive_name = f'{category}_checkpoint_{timestamp}.json'
                archive_path = os.path.join(self.archive_dir, archive_name)
                
                shutil.copy2(checkpoint_path, archive_path)
                os.remove(checkpoint_path)
                
                self.logger.info(f"Checkpoint archived to: {archive_name}")
                
            # Reset processed rows
            self.processed_rows = set()
            
        except Exception as e:
            self.logger.error(f"Error cleaning up checkpoints: {str(e)}")

    def cleanup_logs(self, category: str) -> None:
        """
        Archive logs after category completion
        
        Args:
            category (str): Category name being completed
        """
        try:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            log_path = 'qca_generator.log'
            
            if os.path.exists(log_path):
                # Archive current log
                archive_name = f'{category}_qca_generator_{timestamp}.log'
                archive_path = os.path.join(self.archive_dir, archive_name)
                
                shutil.copy2(log_path, archive_path)
                
                # Clear current log file
                with open(log_path, 'w') as f:
                    f.write(f"Log reset after processing {category} dataset\n")
                
                self.logger.info(f"Logs archived to: {archive_name}")
                
        except Exception as e:
            self.logger.error(f"Error cleaning up logs: {str(e)}")

    def cleanup_progress_files(self, category: str) -> None:
        """
        Clean up intermediate progress files
        
        Args:
            category (str): Category name being completed
        """
        try:
            progress_files = [f for f in os.listdir(self.output_dir) 
                              if f.startswith(f'{category}_qca_progress_')]
            
            for file in progress_files:
                file_path = os.path.join(self.output_dir, file)
                archive_path = os.path.join(self.archive_dir, file)
                
                shutil.move(file_path, archive_path)
                
            self.logger.info(f"Archived {len(progress_files)} progress files")
            
        except Exception as e:
            self.logger.error(f"Error cleaning up progress files: {str(e)}")

    @backoff.on_exception(
        backoff.expo,
        Exception,
        max_tries=3,
        on_backoff=lambda details: logging.warning(f"Retry attempt {details['tries']}")
    )
    def generate_prompt(self, row: pd.Series) -> str:
        """
        Generate prompt for LLMs using Chain of Thought
        
        Args:
            row (pd.Series): Row of data containing article information
            
        Returns:
            str: Generated prompt for LLM
            
        Raises:
            ValueError: If no factual sentences are available
        """
        try:
            # Extract metadata with error checking
            metadata = {
                'source': str(row.get('source', 'Unknown Source')),
                'date': str(row.get('date', 'Unknown Date')),
                'county': str(row.get('county', 'Unknown County')),
                'state': str(row.get('state', 'Unknown State')),
                'title': str(row.get('title', 'Unknown Title'))
            }
            
            # Get factual sentences
            factual_sentences = json.loads(row['factual_content'])
            if not factual_sentences:
                raise ValueError("No factual sentences available")

            # Build prompt with updated requirements based on QCA guidelines
            prompt_parts = [
                "=============================================================================",
                "LOCAL NEWS QA GENERATION PROMPT  •  VERSION 2025-06-REVISED",
                "=============================================================================",
                "You are given a local news article. From it, you will generate ONE QA pair that is:",
                "",
                "    • Clear, locally grounded, and answerable with one correct response.",
                "    • Either:",
                "        ◦ Track A — Fact-based Local QA → grounded in verifiable, non-subjective local information",
                "        ◦ Track B — Community Insight QA → reflects clearly shared local community experiences",
                "",
                "Use the structure below. Do NOT fabricate information.",
                "",
                "─────────────────────────────────────────────────────────────────────────────",
                "ARTICLE DETAILS",
                "─────────────────────────────────────────────────────────────────────────────",
                f"Article Title: {metadata['title']}",
                "",
                "Article Information:",
                f"- Date: {metadata['date']}",
                f"- County: {metadata['county']}",
                f"- State: {metadata['state']}",
                f"- Source: {metadata['source']}",
                "",
                "Article Content (Factual Sentences):"
            ]
            
            # Add sentences
            prompt_parts.extend([f"  {i+1}. {sent}" for i, sent in enumerate(factual_sentences)])
            
            # Add requirements
            prompt_parts.extend([
                "",
                "─────────────────────────────────────────────────────────────────────────────",
                "STRICT QA RULES",
                "─────────────────────────────────────────────────────────────────────────────",
                "1. FACT-BASED ONLY: Question must have ONLY **one correct answer**, and the **only answer** is clearly supported by the article content. No subjectivity.",
                "• Good Example: \"What program was paused in March 2024?\" → ✅",
                "• NOT: \"What platform was recommended?\" or \"What did residents suggest?\" → ✘",
                "• Include time/place qualifiers for precision.",
                "",
                "2. AVOID these:",
                "• Questions asking how people felt, what they recommended, or vague \"why\" questions.",
                "• Any framing that implies subjectivity, speculation, or unconfirmed information.",
                "• Questions about the news article itself (e.g., \"What does the article discuss?\", \"according to the article\", etc.).",
                "",
                "3. CONTEXT must:",
                "• Be 2–4 neutral sentences summarizing article purpose.",
                "• Mention exact date (or month/year), county, and state.",
                "• Attribute information sources (\"The article reported…\", \"Officials noted…\").",
                "• Do **not** leak answer.",
                "",
                "4. ANSWER must:",
                "• Track A: one exact, verifiable fact from the article.",
                "• Track B: a consensus summary if multiple sources echoed same view.",
                "",
                "5. Track A is **preferred**. Only use Track B if consensus is unambiguous.",
                "",
                "6. No meta-content (sources, journalists, publication details).",
                "",
                "7. Privacy & Safety: no doxxing, no private phone numbers.",
                "",
                "─────────────────────────────────────────────────────────────────────────────",
                "OUTPUT FORMAT",
                "─────────────────────────────────────────────────────────────────────────────",
                "[PAIR]",
                "Question: <one clearly answerable question>",
                "Context: <neutral summary with time and location>",
                "Answer: <fact or consensus summary>",
                "Selected Sentences: <e.g., 3, 7, 12>",
                "Pair_type: <fact|insight>",
                "",
                "─────────────────────────────────────────────────────────────────────────────",
                "EXAMPLES OF ACCEPTABLE QUESTIONS",
                "─────────────────────────────────────────────────────────────────────────────",
                "• What city program was in contract phase in March 2024 in Broomfield, Colorado?",
                "• Which consulting firm was hired in June 2025 to evaluate Athens' zoning?",
                "• What public health concern was identified in Honolulu in 2024?",
                "✘ NOT: What do residents want? What do recent discussions say? Which business do people like?",
                "",
                "─────────────────────────────────────────────────────────────────────────────",
                "HARD STOPS",
                "─────────────────────────────────────────────────────────────────────────────",
                "✘ Do NOT generate a QA if:",
                "    - The answer is not clearly stated or verifiable",
                "    - No single correct answer can be derived",
                "    - Content is entirely opinion-based or speculative",
                "",
                "============================================================================="
            ])
            
            return '\n'.join(prompt_parts)
            
        except Exception as e:
            self.logger.error(f"Error generating prompt: {str(e)}")
            raise

    @backoff.on_exception(backoff.expo, Exception, max_tries=3)
    def get_claude_response(self, prompt: str) -> str:
        """
        Get response from Claude API with rate limiting
        
        Args:
            prompt (str): Prompt to send to Claude
            
        Returns:
            str: Claude response
        """
        # Simple rate limiting
        elapsed = time.time() - self.last_api_call
        if elapsed < self.api_rate_limit:
            time.sleep(self.api_rate_limit - elapsed)
            
        try:
            # response = self.anthropic_client.messages.create(
            response = self.openai_client.chat.completions.create(
                model = "o3",
                temperature = 0.7,
                max_tokens = 200,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant that generates high-quality question-context-answer pairs from local news articles."},
                    {"role": "user", "content": prompt}
                ],
            )
            self.last_api_call = time.time()
            return response.choices[0].message.content.strip()
            
        except Exception as e:
            self.logger.error(f"Error getting Claude response: {str(e)}")
            raise

    def parse_claude_response(self, response: str) -> Optional[Dict[str, str]]:
        """
        Parse Claude response and return only the first QCA pair.
        
        Args:
            response (str): Raw response from Claude
            
        Returns:
            Optional[Dict[str, str]]: The first parsed QCA pair or None if parsing fails
        """
        try:
            current_pair = {}
            
            # Basic splitting logic
            for line in response.split('\n'):
                line = line.strip()
                if not line:
                    continue
                    
                if '[PAIR' in line and current_pair:
                    # Return the first found pair immediately
                    return current_pair
                    
                elif line.startswith('Question:'):
                    current_pair['question'] = line[len('Question:'):].strip()
                elif line.startswith('Context:'):
                    current_pair['context'] = line[len('Context:'):].strip()
                elif line.startswith('Answer:'):
                    current_pair['answer'] = line[len('Answer:'):].strip()
                elif line.startswith('Selected Sentences:'):
                    current_pair['selected_sentences'] = line[len('Selected Sentences:'):].strip()
                elif line.startswith('Pair_type:'):
                    current_pair['pair_type'] = line[len('Pair_type:'):].strip()
            
            # Return the first valid pair if one exists
            if current_pair:
                return current_pair

            self.logger.warning("No valid QCA pair found in Claude response")
            return None

        except Exception as e:
            self.logger.error(f"Error parsing Claude response: {str(e)}")
            return None

    def create_refinement_message(self, failed_requirements: List[str]) -> str:
        """
        Create a refinement message explaining which requirements weren't met
        
        Args:
            failed_requirements (List[str]): List of requirement names that failed
            
        Returns:
            str: Refinement message for the LLM
        """
        # Get requirement descriptions from the analyzer
        requirement_descriptions = self.qca_analyzer.get_requirement_descriptions()
        
        message = "The QA pair you provided does not meet the following requirements:\n\n"
        
        for req in failed_requirements:
            if req in requirement_descriptions:
                message += f"• {req.replace('_', ' ').title()}: {requirement_descriptions[req]}\n"
        
        message += "\nPlease revise the QA pair to meet all requirements. Keep your output in the same format:\n\n"
        message += "[PAIR]\nQuestion: <revised question>\nContext: <revised context>\nAnswer: <revised answer>\nSelected Sentences: <sentence numbers>\nPair_type: <fact|insight>"
        
        return message
        
    def process_single_row(self, row_data: Tuple[int, pd.Series]) -> Optional[Dict[str, Any]]:
        """
        Process a single row of data: generate prompt, call Claude, parse response,
        and iteratively refine using QCAMetricAnalyzer if requirements are not met.
        
        Args:
            row_data (Tuple[int, pd.Series]): (index, row)
            
        Returns:
            Optional[Dict[str, Any]]: Result dictionary with QCA pairs, or None on error.
        """
        idx, row = row_data
        try:
            # Skip if already processed
            if idx in self.processed_rows:
                self.logger.info(f"Skipping already processed row {idx}")
                return None

            base_prompt = self.generate_prompt(row)
            result = {'idx': idx}
            
            attempt = 0
            max_attempts = 3
            
            while attempt < max_attempts:
                try:
                    # Get Claude response
                    claude_raw_response = self.get_claude_response(base_prompt)
                    result[f'claude_raw_response_attempt_{attempt + 1}'] = claude_raw_response
                    
                    # Parse response
                    pair = self.parse_claude_response(claude_raw_response)

                    if not pair or not isinstance(pair, dict):
                        self.logger.warning(f"No valid QCA pair found for row {idx}, attempt {attempt + 1}. Retrying...")
                        attempt += 1
                        continue
                    
                    # Extract fields safely
                    question = pair.get('question', '')
                    context = pair.get('context', '')
                    answer = pair.get('answer', '')
                    selected_sentences = pair.get('selected_sentences', '')
                    pair_type = pair.get('pair_type', 'fact')

                    # Check requirements using the new QCAMetricAnalyzer
                    try:
                        requirements_results = self.qca_analyzer.analyze_qa_pair(
                            question=question,
                            context=context,
                            answer=answer
                        )
                        
                        # Check if any requirement failed
                        failed_requirements = [req for req, result_val in requirements_results.items() 
                                             if result_val.lower() == "no"]
                        
                        if len(failed_requirements) < 1:
                            result['claude_Question_1'] = question
                            result['claude_QuestionContext_1'] = context
                            result['claude_Answer_1'] = answer
                            result['claude_SelectedSentences_1'] = selected_sentences
                            result['claude_PairType_1'] = pair_type
                            result['requirements_results'] = str(requirements_results)
                            result['failed_requirements_count'] = len(failed_requirements)
                            
                            # Calculate overall score using the analyzer
                            overall_score = self.qca_analyzer.calculate_overall_score(requirements_results)
                            result['overall_quality_score'] = overall_score
                            
                            if failed_requirements:
                                result['tolerated_failed_requirements'] = str(failed_requirements)
                            
                            self.processed_rows.add(idx)
                            tolerance_msg = "with perfect requirements" if len(failed_requirements) == 0 else f"with {len(failed_requirements)} tolerated failed requirement: {failed_requirements}"
                            self.logger.info(f"Row {idx} completed successfully after {attempt + 1} attempt(s) {tolerance_msg}")
                            return result
                        else:
                            # Too many requirements failed - create refinement message
                            self.logger.info(f"Row {idx}, attempt {attempt + 1}: {len(failed_requirements)} failed requirements (tolerance threshold: 1): {failed_requirements}")
                            
                            # Update prompt for next attempt
                            refinement_message = self.create_refinement_message(failed_requirements)
                            base_prompt += f"\n\n{refinement_message}"
                            
                            # Store failed requirements for debugging
                            result[f'failed_requirements_attempt_{attempt + 1}'] = str(failed_requirements)
                            result[f'requirements_results_attempt_{attempt + 1}'] = str(requirements_results)
                            
                    except Exception as e:
                        self.logger.error(f"Error in QCA analysis for row {idx}, attempt {attempt + 1}: {str(e)}")
                        # Continue to next attempt on analysis error
                        
                except Exception as e:
                    self.logger.error(f"Error with Claude model for row {idx}, attempt {attempt + 1}: {str(e)}")

                attempt += 1
            
            self.logger.warning(f"Skipping row {idx} after {max_attempts} attempts - more than 1 requirement failed (tolerance threshold exceeded)")
            return None
        
        except Exception as e:
            self.logger.error(f"Error processing row {idx}: {str(e)}")
            self.error_queue.put((idx, str(e)))
        return None

    def process_batch(self, batch_data: List[Tuple[int, pd.Series]]) -> None:
        """
        Process a batch of rows in parallel and put results in queue
        
        Args:
            batch_data (List[Tuple[int, pd.Series]]): List of (index, row) tuples to process
        """
        # Parallel execution with ThreadPoolExecutor
        with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
            futures = [
                executor.submit(self.process_single_row, row_data) 
                for row_data in batch_data
            ]
            
            for future in concurrent.futures.as_completed(futures):
                try:
                    result = future.result()
                    if result:
                        self.result_queue.put(result)
                except Exception as e:
                    self.logger.error(f"Error in batch processing: {str(e)}")

    def save_progress(self, df: pd.DataFrame, category: str, timestamp: str) -> None:
        """
        Save intermediate progress with error handling
        
        Args:
            df (pd.DataFrame): DataFrame to save
            category (str): Category name
            timestamp (str): Timestamp for the file name
        """
        try:
            output_path = os.path.join(
                self.output_dir, 
                f'{category}_qca_progress_{timestamp}.parquet'
            )
            df.to_parquet(output_path)
            self.logger.info(f"Progress saved to: {output_path}")
            
            # Save checkpoint
            self.save_checkpoint()
            
        except Exception as e:
            self.logger.error(f"Error saving progress: {str(e)}")

    def process_dataset(self, df: pd.DataFrame, category: str) -> pd.DataFrame:
        """
        Process dataset with batch processing and parallel execution
        
        Args:
            df (pd.DataFrame): Input DataFrame to process
            category (str): Category name
            
        Returns:
            pd.DataFrame: Processed DataFrame with QCA pairs
        """
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        processed_df = df.copy()
        
        # Initialize columns for QCA pairs and analysis results
        base_columns = ['Question', 'QuestionContext', 'Answer', 'SelectedSentences', 'PairType']
        for i in range(1, 3):
            for col in base_columns:
                processed_df[f'claude_{col}_{i}'] = ''
        
        # Add columns for requirements analysis and tolerance tracking
        processed_df['requirements_results'] = ''
        processed_df['failed_requirements_count'] = 0
        processed_df['overall_quality_score'] = 0.0
        
        # Initialize columns for attempt tracking (up to max_attempts)
        max_attempts = 3
        for attempt in range(1, max_attempts + 1):
            processed_df[f'claude_raw_response_attempt_{attempt}'] = ''
            processed_df[f'failed_requirements_attempt_{attempt}'] = ''
            processed_df[f'requirements_results_attempt_{attempt}'] = ''

        # Process in batches
        batch_size = 32
        for start_idx in tqdm(range(0, len(processed_df), batch_size)):
            end_idx = min(start_idx + batch_size, len(processed_df))
            batch_data = list(processed_df.iloc[start_idx:end_idx].iterrows())
            
            # Process batch in parallel
            self.process_batch(batch_data)
            
            # Collect results
            while not self.result_queue.empty():
                result = self.result_queue.get()
                idx = result.pop('idx')
                for col, value in result.items():
                    # Handle different value types safely
                    if isinstance(value, (list, dict)):
                        # Convert complex data structures to string for storage
                        processed_df.at[idx, col] = str(value)
                    else:
                        processed_df.at[idx, col] = value
            
            # Handle any errors
            while not self.error_queue.empty():
                idx, error = self.error_queue.get()
                self.logger.warning(f"Error in row {idx}: {error}")
            
            # Save progress periodically
            if (start_idx + batch_size) % 50 == 0:
                self.save_progress(processed_df, category, timestamp)
        
        # Save final results
        final_path = os.path.join(
            self.output_dir, 
            f'{category}_qca_final_{timestamp}.parquet'
        )
        processed_df.to_parquet(final_path)
        self.logger.info(f"Final results saved to: {final_path}")
        
        # Print statistics
        self.print_statistics(processed_df, category)
        
        return processed_df

    def print_statistics(self, df: pd.DataFrame, category: str) -> None:
        """
        Print basic statistics for the Claude QCA generation
        """
        try:
            print(f"\nProcessing Statistics for {category}:")
            print("-" * 50)
            print(f"Total rows processed: {len(df)}")
            
            # Count how many rows have a non-empty question in at least the first pair
            if 'claude_Question_1' in df.columns:
                completed_rows = sum(df['claude_Question_1'].astype(str).str.len() > 0)
                success_rate = (completed_rows / len(df)) * 100
                
                print(f"\nClaude Model Statistics:")
                print(f"Successfully completed: {completed_rows}")
                print(f"Success rate: {success_rate:.2f}%")
                
                # Quality score statistics
                if 'overall_quality_score' in df.columns and completed_rows > 0:
                    completed_mask = df['claude_Question_1'].astype(str).str.len() > 0
                    quality_scores = pd.to_numeric(df.loc[completed_mask, 'overall_quality_score'], errors='coerce')
                    avg_quality = quality_scores.mean()
                    print(f"Average quality score: {avg_quality:.3f}")
                    
                # Average lengths - only calculate for non-empty entries
                non_empty_mask = df['claude_Question_1'].astype(str).str.len() > 0
                if non_empty_mask.any():
                    avg_question = df.loc[non_empty_mask, 'claude_Question_1'].str.len().mean()
                    avg_context = df.loc[non_empty_mask, 'claude_QuestionContext_1'].str.len().mean()
                    avg_answer = df.loc[non_empty_mask, 'claude_Answer_1'].str.len().mean()
                        
                    print("\nAverage lengths:")
                    print(f"Questions: {avg_question:.1f} characters")
                    print(f"Context: {avg_context:.1f} characters")
                    print(f"Answers: {avg_answer:.1f} characters")
                
                # Requirements analysis statistics
                if 'requirements_results' in df.columns:
                    requirements_with_results = df[df['requirements_results'] != ''].copy()
                    if len(requirements_with_results) > 0:
                        print(f"\nRequirements Analysis:")
                        print(f"Rows with requirements analysis: {len(requirements_with_results)}")
            else:
                print("No QCA columns found in DataFrame")
                    
            self.logger.info(f"Processing completed for {category}")
            
        except Exception as e:
            self.logger.error(f"Error calculating statistics: {str(e)}")

def main():
    """Main execution function"""
    # Initialize generator
    generator = QCAGenerator(config)
    
    try:
        # Load the most recent factual data
        factual_dir = 'processed_data/factual_content'
        files = {
            'merged': sorted([f for f in os.listdir(factual_dir) if f.startswith('merged_factual_')], reverse=True)
        }
        
        if not all(files.values()):
            raise FileNotFoundError("Could not find factual data files")
        
        # Load and process datasets
        results = {}
        for category, file_list in files.items():
            logging.info(f"Processing {category} dataset...")
            df = pd.read_parquet(os.path.join(factual_dir, file_list[0]))

            print(f"\nProcessing {category} dataset with {len(df)} rows...")
            # remove duplicates based on 'local_id' column
            df = df.drop_duplicates(subset=['local_id'])
            print(f"After removing duplicates, {len(df)} rows remain.")

            results[category] = generator.process_dataset(df, category)

            # Clean up after processing
            generator.cleanup_checkpoints(category)
            generator.cleanup_logs(category)
            generator.cleanup_progress_files(category)
            
            logging.info(f"Completed processing and cleanup for {category} dataset")
        
        return results['merged']
        
    except Exception as e:
        logging.error(f"Error in main execution: {str(e)}")
        raise

def run_qca_generation():
    """Entry point function with error handling"""
    try:
        logging.info("Starting QCA generation process...")
        processed_merged = main()
        logging.info("QCA generation completed successfully")
        return processed_merged
        
    except Exception as e:
        logging.error(f"Fatal error in QCA generation: {str(e)}")
        raise

processed_merged = run_qca_generation()

## Census QA Generation

In [ ]:
filtered_counties = pd.read_csv('balanced_rucc_counties.csv', encoding='cp1252')
localness_metrics_mapping = pd.read_csv('localness_metrics_mapping.csv')

# First, let's identify the metric columns
id_columns = ['STATE_NAME', 'COUNTY_NAME', 'fips', 'RUCC', 'RUCC_group', 'POP_COU','POPPCT_RUR']
metric_columns = [col for col in filtered_counties.columns if col not in id_columns]

# Create a dictionary mapping metric to its description
metric_descriptions = {}
for _, row in localness_metrics_mapping.iterrows():
    metric_descriptions[row['Column_name']] = row['Metrics']

# Create new columns for QA pairs
for metric in metric_columns:
    filtered_counties[f'{metric}_q1'] = ''
    filtered_counties[f'{metric}_a1'] = np.nan
    filtered_counties[f'{metric}_q2'] = ''
    filtered_counties[f'{metric}_a2'] = False
    filtered_counties[f'{metric}_q3'] = ''
    filtered_counties[f'{metric}_a3'] = False

# Log for NaN values
log_nan_metrics = []

# Set random seed for reproducibility (optional)
random.seed(42)
np.random.seed(42)

# Process each row with progress bar
print("Processing counties...")
for idx, row in tqdm(filtered_counties.iterrows(), total=len(filtered_counties), desc="Counties", unit="county"):
    county_name = row['COUNTY_NAME']
    state_name = row['STATE_NAME']
    
    # Process each metric with nested progress bar
    for metric in tqdm(metric_columns, desc=f"{county_name}, {state_name}", leave=False, unit="metric"):
        metric_value = row[metric]
        
        # Skip if value is NaN
        if pd.isna(metric_value):
            log_nan_metrics.append(f"Skipped {metric} for {county_name}, {state_name} due to NaN value")
            continue
        
        # Skip if metric doesn't have a description
        if metric not in metric_descriptions:
            log_nan_metrics.append(f"Skipped {metric} for {county_name}, {state_name} due to missing description")
            continue
        
        description = metric_descriptions[metric]
        
        # Generate Q1 (fill-in-the-blank)
        modified_description = description.replace('this county', f'{county_name}, {state_name}')
        # modified_description = modified_description.replace('county', f'{county_name}, {state_name}')
        
        if modified_description.endswith('.'):
            modified_description = modified_description[:-1]
        q1 = f"{modified_description} is []"
        a1 = metric_value
        
        filtered_counties.at[idx, f'{metric}_q1'] = q1
        filtered_counties.at[idx, f'{metric}_a1'] = a1
        
        # Find counties for comparison
        other_counties = filtered_counties[filtered_counties.index != idx]
        valid_other_counties = other_counties[other_counties[metric].notna()]
        higher_counties = valid_other_counties[valid_other_counties[metric] > metric_value]
        lower_counties = valid_other_counties[valid_other_counties[metric] < metric_value]
        
        # Generate comparison questions
        modified_comparison = description.replace('this county', f'{county_name} County, {state_name}')
        # modified_comparison = modified_comparison.replace('county', f'{county_name}, {state_name}')
        if modified_comparison.endswith('.'):
            modified_comparison = modified_comparison[:-1]
        
        # Generate Q2 (comparison - normally higher than lower value)
        if len(higher_counties) > 0:
            # Random selection from higher counties to avoid bias
            higher_county = higher_counties.sample(n=1).iloc[0]
            county_state_1 = f"{higher_county['COUNTY_NAME']} County, {higher_county['STATE_NAME']}"
            q2 = f"{modified_comparison} is higher than that in {county_state_1}"
            a2 = False  # Current is not higher than a higher county
        elif len(lower_counties) > 0:
            # Edge case: highest value, generate like Q3
            lower_county = lower_counties.sample(n=1).iloc[0]
            county_state_1 = f"{lower_county['COUNTY_NAME']} County, {lower_county['STATE_NAME']}"
            q2 = f"{modified_comparison} is higher than that in {county_state_1}"
            a2 = True  # Current is higher than a lower county
        else:
            # No comparison possible
            q2 = "Unable to generate comparison question (no other counties with valid values)"
            a2 = False
        
        filtered_counties.at[idx, f'{metric}_q2'] = q2
        filtered_counties.at[idx, f'{metric}_a2'] = a2
        
        # Generate Q3 (comparison - normally higher than higher value)
        if len(lower_counties) > 0:
            # Random selection from lower counties to avoid bias
            lower_county = lower_counties.sample(n=1).iloc[0]
            county_state_2 = f"{lower_county['COUNTY_NAME']}, {lower_county['STATE_NAME']}"
            q3 = f"{modified_comparison} is higher than that in {county_state_2}"
            a3 = True  # Current is higher than a lower county
        elif len(higher_counties) > 0:
            # Edge case: lowest value, generate like Q2
            higher_county = higher_counties.sample(n=1).iloc[0]
            county_state_2 = f"{higher_county['COUNTY_NAME']}, {higher_county['STATE_NAME']}"
            q3 = f"{modified_comparison} is higher than that in {county_state_2}"
            a3 = False  # Current is not higher than a higher county
        else:
            # No comparison possible
            q3 = "Unable to generate comparison question (no other counties with valid values)"
            a3 = False
        
        filtered_counties.at[idx, f'{metric}_q3'] = q3
        filtered_counties.at[idx, f'{metric}_a3'] = a3

In [ ]:
def transform_census_data(census_df):
    """
    Transform census data to create question/answer format with random sampling
    
    Args:
        census_df: DataFrame with census data containing q1/q2/q3 and a1/a2/a3 columns
    
    Returns:
        DataFrame with transformed data
    """
    
    # Set random seed for reproducibility
    np.random.seed(42)
    random.seed(42)
    
    # Identify base columns (those that have _q1, _q2, _q3 variants)
    all_columns = census_df.columns.tolist()
    
    # Find all unique metric names that have question variants
    metric_names = set()
    for col in all_columns:
        if col.endswith('_q1') or col.endswith('_q2') or col.endswith('_q3'):
            # Remove the _q1, _q2, _q3 suffix to get base metric name
            base_name = col.rsplit('_q', 1)[0]
            metric_names.add(base_name)
    
    metric_names = list(metric_names)
    print(f"Found {len(metric_names)} metrics with question variants")
    
    # Create the transformed dataframe
    transformed_rows = []
    
    for idx, row in census_df.iterrows():
        # Keep the base identifying columns
        base_row = {
            'STATE_NAME': row['STATE_NAME'],
            'COUNTY_NAME': row['COUNTY_NAME'],
            'fips': row['fips'],
            'POP_COU': row['POP_COU'],
            'RUCC': row['RUCC'],
            'RUCC_group': row['RUCC_group'],
        }
        
        # For each metric, randomly select q1, q2, or q3
        for metric in metric_names:
            # Check if all required columns exist for this metric
            q1_col = f"{metric}_q1"
            q2_col = f"{metric}_q2"
            q3_col = f"{metric}_q3"
            a1_col = f"{metric}_a1"
            a2_col = f"{metric}_a2"
            a3_col = f"{metric}_a3"
            
            # Skip if any required columns are missing
            required_cols = [q1_col, q2_col, q3_col, a1_col, a2_col, a3_col]
            if not all(col in census_df.columns for col in required_cols):
                continue
            
            # Randomly select question type with specified probabilities
            # q1: 50%, q2: 25%, q3: 25%
            question_type = np.random.choice(['q1', 'q2', 'q3'], p=[0.5, 0.25, 0.25])
            
            # Get the selected question and answer
            if question_type == 'q1':
                question_text = row[q1_col]
                answer_text = row[a1_col]
                q_type = 'numerical'
            elif question_type == 'q2':
                question_text = row[q2_col]
                answer_text = row[a2_col]
                q_type = 'text'
            else:  # q3
                question_text = row[q3_col]
                answer_text = row[a3_col]
                q_type = 'text'
            
            # Create a row for this metric
            metric_row = base_row.copy()
            metric_row.update({
                'metric': metric,
                'question_type': q_type,
                'question': question_text,
                'answer': answer_text,
                'selected_variant': question_type
            })
            
            transformed_rows.append(metric_row)
    
    # Create the transformed dataframe
    transformed_df = pd.DataFrame(transformed_rows)
    
    print(f"Original dataframe shape: {census_df.shape}")
    print(f"Transformed dataframe shape: {transformed_df.shape}")
    print(f"Number of metrics per county: {len(metric_names)}")
    
    # Show distribution of question types
    type_distribution = transformed_df['question_type'].value_counts(normalize=True)
    print("\nQuestion type distribution:")
    print(type_distribution)
    
    return transformed_df


sampled_df = pd.read_csv('census_processed_data')
transformed_df = transform_census_data(sampled_df)